# Ensemble Learning Techniques Code Companion

This notebook connects ensemble theory with code: voting, bagging, random forests, and boosting. It keeps the focus on how predictions are combined.


In [ ]:
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier, BaggingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


## Load a Classification Dataset

All models will solve the same classification problem so their behavior can be compared fairly.


In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

print("Classes:", dict(enumerate(data.target_names)))
print("Rows and features:", X.shape)
X.head()


## Split the Data


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## Create Different Base Models

Voting works better when the models are different from each other. Here Logistic Regression, KNN, and Decision Tree learn in different ways.


In [ ]:
logistic = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

knn = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=7))
])

tree = DecisionTreeClassifier(max_depth=4, random_state=42)


## Hard Voting

Hard voting combines final class labels. Each model votes for a class, and the majority class wins.


In [ ]:
hard_voting = VotingClassifier(
    estimators=[("logistic", logistic), ("knn", knn), ("tree", tree)],
    voting="hard"
)

hard_voting.fit(X_train, y_train)
y_pred_hard = hard_voting.predict(X_test)

print("Hard voting accuracy:", accuracy_score(y_test, y_pred_hard))
print("Hard voting F1:", f1_score(y_test, y_pred_hard))


## Soft Voting

Soft voting combines probabilities. It can use confidence, not only final class labels.


In [ ]:
soft_voting = VotingClassifier(
    estimators=[("logistic", logistic), ("knn", knn), ("tree", tree)],
    voting="soft"
)

soft_voting.fit(X_train, y_train)
y_pred_soft = soft_voting.predict(X_test)
y_proba_soft = soft_voting.predict_proba(X_test)[:, 1]

print("Soft voting accuracy:", accuracy_score(y_test, y_pred_soft))
print("Soft voting F1:", f1_score(y_test, y_pred_soft))
print("Soft voting ROC-AUC:", roc_auc_score(y_test, y_proba_soft))


## Bagging

Bagging trains many versions of a high-variance model on different samples of data and combines their predictions.


In [ ]:
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    random_state=42,
    n_jobs=1
)

bagging.fit(X_train, y_train)
y_pred_bagging = bagging.predict(X_test)

print("Bagging accuracy:", accuracy_score(y_test, y_pred_bagging))
print("Bagging F1:", f1_score(y_test, y_pred_bagging))


## Random Forest

Random Forest is bagging plus random feature selection. It is included here to show where it fits inside ensemble learning.


In [ ]:
forest = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    random_state=42,
    n_jobs=1
)

forest.fit(X_train, y_train)
y_pred_forest = forest.predict(X_test)
y_proba_forest = forest.predict_proba(X_test)[:, 1]

print("Random Forest accuracy:", accuracy_score(y_test, y_pred_forest))
print("Random Forest F1:", f1_score(y_test, y_pred_forest))
print("Random Forest ROC-AUC:", roc_auc_score(y_test, y_proba_forest))


## Boosting

Boosting trains models sequentially. Each new model focuses more on examples that earlier models handled poorly.


In [ ]:
boosting = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)

boosting.fit(X_train, y_train)
y_pred_boosting = boosting.predict(X_test)
y_proba_boosting = boosting.predict_proba(X_test)[:, 1]

print("Boosting accuracy:", accuracy_score(y_test, y_pred_boosting))
print("Boosting F1:", f1_score(y_test, y_pred_boosting))
print("Boosting ROC-AUC:", roc_auc_score(y_test, y_proba_boosting))


## Compare Ensemble Results


In [ ]:
results = pd.DataFrame([
    {"model": "Hard Voting", "accuracy": accuracy_score(y_test, y_pred_hard), "f1": f1_score(y_test, y_pred_hard), "roc_auc": None},
    {"model": "Soft Voting", "accuracy": accuracy_score(y_test, y_pred_soft), "f1": f1_score(y_test, y_pred_soft), "roc_auc": roc_auc_score(y_test, y_proba_soft)},
    {"model": "Bagging", "accuracy": accuracy_score(y_test, y_pred_bagging), "f1": f1_score(y_test, y_pred_bagging), "roc_auc": None},
    {"model": "Random Forest", "accuracy": accuracy_score(y_test, y_pred_forest), "f1": f1_score(y_test, y_pred_forest), "roc_auc": roc_auc_score(y_test, y_proba_forest)},
    {"model": "Boosting", "accuracy": accuracy_score(y_test, y_pred_boosting), "f1": f1_score(y_test, y_pred_boosting), "roc_auc": roc_auc_score(y_test, y_proba_boosting)}
])

results
